[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/17_sequence_and_state_space.ipynb)

# 17. Sequence recurrence and state-space updates

RNN 상태 업데이트에서 gated recurrence, linear state-space recurrence, scan, selective state update로 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Vanilla recurrence

h_t = tanh(A h_{t-1} + B x_t)를 직접 반복한다.


In [ ]:
xs = torch.tensor([[1.], [2.], [3.]], device=device)
A = torch.tensor([[0.5]], device=device)
B = torch.tensor([[1.0]], device=device)

h = torch.zeros(1, device=device)
for t, x in enumerate(xs):
    h = torch.tanh(A @ h + B @ x)
    print(t, h)


In [ ]:
_ = profile_call("one RNN update", lambda h_, x_: torch.tanh(A@h_ + B@x_), torch.zeros(1,device=device), xs[0])


## 2. GRU gate idea

update gate가 old/new state의 혼합 비율을 결정한다.


In [ ]:
h = torch.tensor([0.2, -0.1], device=device)
candidate = torch.tensor([0.8, 0.5], device=device)
z = torch.sigmoid(torch.tensor([1.0, -1.0], device=device))

h_new = (1-z)*h + z*candidate
print("gate:", z)
print("new h:", h_new)


In [ ]:
_ = profile_call("GRU mixing", lambda: (1-z)*h + z*candidate)


## 3. Linear SSM recurrence

nonlinearity 없이 A h + B x를 반복한다.


In [ ]:
A = torch.tensor([[0.8, 0.1], [0.0, 0.9]], device=device)
B = torch.tensor([[1.0], [0.5]], device=device)
h = torch.zeros(2, device=device)

for x in [1.0, 0.0, -1.0]:
    h = A @ h + B[:, 0] * x
    print(h)


In [ ]:
_ = profile_call("linear SSM step", lambda z: A@z + B[:,0], torch.zeros(2,device=device))


## 4. Associative scan composition

affine recurrence h'=a*h+b를 pair composition으로 묶을 수 있음을 본다.


In [ ]:
pairs = [
    (torch.tensor(0.5, device=device), torch.tensor(1.0, device=device)),
    (torch.tensor(0.8, device=device), torch.tensor(2.0, device=device)),
]

a1, b1 = pairs[0]
a2, b2 = pairs[1]

a_comp = a2 * a1
b_comp = a2 * b1 + b2

h0 = torch.tensor(3.0, device=device)
sequential = a2 * (a1 * h0 + b1) + b2
composed = a_comp * h0 + b_comp

print(sequential, composed)


In [ ]:
_ = profile_call("affine scan composition", lambda: (a2*a1, a2*b1+b2))


## 5. Selective state update

input에 따라 decay/gate가 달라지는 selective recurrence를 본다.


In [ ]:
h = torch.zeros(2, device=device)
inputs = torch.tensor([[1., -1.], [0.5, 2.], [-1., 0.]], device=device)

for x in inputs:
    gate = torch.sigmoid(x)
    h = gate * h + (1 - gate) * x
    print("gate", gate, "state", h)


In [ ]:
def selective_update(h_, x_):
    gate_ = torch.sigmoid(x_)
    return gate_ * h_ + (1 - gate_) * x_

_ = profile_call(
    "selective state update",
    selective_update,
    torch.zeros(2, device=device),
    inputs[0],
)


## References and provenance

**[17.1] RNN/GRU/LSTM**
- 출처: classical recurrent networks
- 이 노트북에서 가져온 부분: state and gated state updates

**[17.2] S4**
- 출처: Gu et al., Efficiently Modeling Long Sequences with Structured State Spaces
- 이 노트북에서 가져온 부분: linear state-space recurrence

**[17.3] Mamba**
- 출처: Gu & Dao, Mamba
- 이 노트북에서 가져온 부분: selective state-space update and scan

**[17.4] Kimi linear/KDA lineage**
- 출처: Kimi linear-attention papers
- 이 노트북에서 가져온 부분: recurrent matrix-state alternatives to full attention
